# 📊 Data, Calculations, Actions

## Grokking Simplicity Applied to LLM Programming

The book *Grokking Simplicity* teaches us to classify all code into three categories:

| Category | Description | LLM Equivalent | Properties |
|----------|-------------|-----------------|------------|
| **DATA** | Inert facts, no behavior | Signatures, Datasets | Pure, serializable, free |
| **CALCULATIONS** | Pure functions, same in → same out | Metrics, Evaluators | Deterministic, testable, instant |
| **ACTIONS** | Side effects, I/O, non-deterministic | LLM Calls, API requests | Costly, slow, may fail |

This separation is the foundation of reliable AI systems:

```
┌─────────────────────────────────────────────┐
│  DATA         │ Signatures, Datasets         │  ← Pure, free, portable
├───────────────┼──────────────────────────────┤
│  CALCULATIONS │ Metrics, Evaluators          │  ← Deterministic, testable
├───────────────┼──────────────────────────────┤
│  ACTIONS      │ LLM Calls, Optimizations     │  ← Costly, non-deterministic
└─────────────────────────────────────────────┘
```

Let's see each layer in action.

In [3]:
import sys
sys.path.insert(0, ".")
import dspy
from dspy_tasks.tasks import get_task, list_by_tier
from dspy_tasks.data import ClassifySentiment, ExtractEntities, SummarizeText
from dspy_tasks.calculations import sentiment_exact_match, entity_f1, summary_quality
from dspy_tasks.actions import run_baseline
from dspy_tasks.visualize import *

## Part 1: DATA — Signatures as Declarations

A DSPy **Signature** is pure data. It declares:
- What **inputs** the task needs
- What **outputs** it should produce
- A **docstring** describing the intent

It says **WHAT** you want, never **HOW** to get it. There are no prompt templates,
no system messages, no few-shot examples baked in. Just a declaration.

You could serialize a Signature to JSON and send it across the network — it has no behavior, no side effects, no dependencies.

In [4]:
# A Signature is pure DATA — it declares intent, not implementation
print("ClassifySentiment fields:")
for name, field in ClassifySentiment.model_fields.items():
    field_info = ClassifySentiment.__annotations__.get(name)
    print(f"  {name}: {field_info}")

display_insight("DATA Principle",
    "Signatures are pure data declarations. They describe WHAT you want, never HOW. "
    "You could serialize them to JSON and send them across the network — they have no behavior.")

ClassifySentiment fields:
  review: <class 'str'>
  sentiment: <class 'str'>


In [5]:
task = get_task("sentiment")
examples = task.load_examples()
print(f"Dataset: {len(examples)} examples\n")
for ex in examples[:5]:
    print(f"  Review: {str(ex.review)[:60]}...")
    print(f"  Sentiment: {ex.sentiment}\n")

Dataset: 25 examples

  Review: Absolutely love this blender! It crushes ice in seconds and ...
  Sentiment: positive

  Review: The laptop arrived with a cracked screen and customer suppor...
  Sentiment: negative

  Review: The coffee maker works as described. Nothing exceptional, no...
  Sentiment: neutral

  Review: Oh great, another pair of wireless earbuds that dies after t...
  Sentiment: negative

  Review: I was skeptical at first, but this standing desk has complet...
  Sentiment: positive



## Part 2: CALCULATIONS — Metrics as Pure Functions

Metrics are **CALCULATIONS** — the second layer of Grokking Simplicity:

- ✅ Same input → always the same output
- ✅ No LLM calls, no network, no side effects
- ✅ Infinitely fast and perfectly deterministic
- ✅ You can test them **right now**, without any API key

This is your **specification** — your "test suite" for AI. If the metric says 0.85, that's the ground truth about quality, regardless of which model or prompt produced the output.

In [6]:
# CALCULATION — No LLM needed! Instantly testable.
class FakeExample:
    sentiment = "positive"
class FakePrediction:
    sentiment = "positive"
class WrongPrediction:
    sentiment = "negative"

score_correct = sentiment_exact_match(FakeExample(), FakePrediction())
score_wrong = sentiment_exact_match(FakeExample(), WrongPrediction())

print(f"Correct prediction score: {score_correct}")   # 1.0
print(f"Wrong prediction score:   {score_wrong}")     # 0.0

display_insight("CALCULATION Principle",
    "Metrics are pure functions. You can test them with fake data, "
    "no API key needed. They're infinitely fast and perfectly deterministic. "
    "This is your specification — your 'test suite' for AI.")

Correct prediction score: 1.0
Wrong prediction score:   0.0


In [7]:
class ExampleEntities:
    entities = "Apple, Google, Microsoft"
class PredEntities:
    entities = "Apple, Microsoft, Amazon"  # Got 2 of 3, added 1 wrong

score = entity_f1(ExampleEntities(), PredEntities())
print(f"Entity F1 score: {score:.3f}")
print("(2 correct out of 3 expected, 1 false positive → F1 reflects both precision and recall)")

Entity F1 score: 0.667
(2 correct out of 3 expected, 1 false positive → F1 reflects both precision and recall)


## Part 3: ACTIONS — LLM Calls as I/O

Now we reach the **ACTIONS** layer — where side effects happen:

- ⚠️ Calls an LLM over the network
- ⚠️ Takes time (seconds to minutes)
- ⚠️ Costs money (tokens aren't free)
- ⚠️ Non-deterministic (same input may give different output)
- ⚠️ Might fail (rate limits, timeouts, outages)

We **isolate** actions so that everything else remains testable, portable, and fast.
When you change models, only the ACTIONS change. The DATA and CALCULATIONS stay exactly the same.

In [8]:
import ipywidgets as widgets
from dspy_tasks.config import get_available_models, get_default_model, configure_dspy

AVAILABLE_MODELS = get_available_models()

model_dd = model_picker(AVAILABLE_MODELS, default=get_default_model())
btn = run_button("Run Sentiment Baseline")
out = widgets.Output()

def on_run(b):
    with out:
        out.clear_output()
        print(f"⏳ Running sentiment task on {model_dd.value}...")
        result = run_baseline("sentiment", model_dd.value, max_eval=10)
        display_score("Baseline Score", result.score)
        print(f"⏱️  {result.elapsed_seconds}s | {result.llm_calls} LLM calls")
        display_results_table(result.individual_scores)

btn.on_click(on_run)
display(widgets.HBox([model_dd, btn]), out)

Output()

In [9]:
# Run all 3 Tier 1 tasks
task_dd = widgets.Dropdown(
    options=[(t.name, t.id) for t in list_by_tier(1)[:3]],
    description="Task:"
)
btn2 = run_button("Run Task")
out2 = widgets.Output()

def on_run2(b):
    with out2:
        out2.clear_output()
        task = get_task(task_dd.value)
        print(f"⏳ Running {task.name} on {model_dd.value}...")
        result = run_baseline(task_dd.value, model_dd.value, max_eval=8)
        display_score(task.name, result.score)
        print(f"⏱️  {result.elapsed_seconds}s | 💡 {task.teaching_point}")
        display_results_table(result.individual_scores[:5])

btn2.on_click(on_run2)
display(widgets.HBox([task_dd, btn2]), out2)

Output()

## 🔑 Summary: The Three Layers

We've now seen all three layers in action:

| Layer | What We Did | Key Insight |
|-------|-------------|-------------|
| **DATA** | Inspected `ClassifySentiment` signature, loaded examples | Pure declarations — no behavior, no cost |
| **CALCULATIONS** | Tested `sentiment_exact_match` and `entity_f1` with fake data | No API key needed — instant, deterministic |
| **ACTIONS** | Ran `run_baseline()` against a real LLM | The only layer with side effects |

This separation gives you a superpower: **you can change any layer independently.**

- Swap the model? Only ACTIONS change.
- Tighten the metric? Only CALCULATIONS change.
- Add a new field to the task? Only DATA changes.

→ **Next up — Notebook 02**: What happens when we make the module *deeper*?
  Chain-of-Thought, few-shot examples, and the first taste of optimization.

In [10]:
display_insight("The Foundation",
    "Signatures and metrics are your source code — they're pure, testable, and portable. "
    "LLM calls are just the I/O layer. When you change models, only the ACTIONS change. "
    "The DATA and CALCULATIONS stay exactly the same.",
    icon="🏗️")